# Assignment 3 — A Variational Autoencoder for MNIST

**CSCI 4052U — Machine Learning II** · **Unit.** `chapter_17_variational_autoencoders`
· **Weight.** 100 marks

Chapter 17 derives the evidence lower bound and assembles it into the variational
autoencoder. This notebook builds that model and runs it on MNIST.

**The pieces are provided.** The convolutional encoder, the convolutional decoder and
the two terms of the loss are all written for you in `components.py`; the training loop
is in `train.py`. Your work is to **assemble** them: wire the encoder and decoder into a
VAE, combine the two loss terms into the objective, run the training function, and
decode.

The latent variable is **two-dimensional**, which is what makes Part 4 possible: a 2-D
latent space can be swept exhaustively, so one picture holds everything the model
learned to draw.

## What to hand in

1. **This notebook, with every cell run and its output visible.** Do not clear the
   outputs — the figures *are* the answer.
2. **Your written answers**, in the two markdown cells marked **Your answer**.

| Part | | Marks |
|---|---|---|
| 1 | Construct the VAE from the components | 30 |
| 2 | Construct the loss from the two components | 25 |
| 3 | Train it, logging both terms | 25 |
| 4 | Decode the precomputed latent sweep | 20 |

A **bonus notebook**, `assignment3_bonus.ipynb`, carries a further 20 marks. It reads
the model this notebook saves in its last cell, so finish this one first.

## Before you start

**Turn on a GPU.** On Colab: *Runtime → Change runtime type → T4 GPU*. Training takes
one to two minutes on a GPU and roughly twenty times that without one.

**Save a copy first.** *File → Save a copy in Drive*.

In [ ]:
# COLAB SETUP -- run this cell first. Outside Colab it does nothing.
import os
import subprocess
import sys
import urllib.request

try:  # find_spec raises when the parent package is absent, so import instead
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab ships torch and torchvision; install only what it lacks.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightning", "pandas"], check=True)


def _fetch(name, base="https://raw.githubusercontent.com/kenpu-uoit/csci4052u-machine-learning-2/main/assignments/assignment3"):
    """Download a support file when it is not already beside the notebook."""
    if not os.path.exists(name):
        urllib.request.urlretrieve(f"{base}/{name}", name)
        print(f"downloaded {name}")


if IN_COLAB:
    _fetch("components.py")
    _fetch("utils.py")
    _fetch("train.py")
    _fetch("z_samples.pt")

print("setup complete")

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms

# The provided building blocks.
from components import D_Z, SIGMA2, Encoder, Decoder
from components import reconstruction_loss, kl_divergence, load_z_samples
from train import train_vae, plot_metrics, decode_z, save_trained
from utils import show_grid, check_vae, check_loss_fn, check_history, check_decoded_grid

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device {DEVICE} | D_z {D_Z} | sigma^2 {SIGMA2}")
if DEVICE == "cpu":
    print("WARNING: no GPU. Part 3 will take roughly 20 minutes. Switch to a T4.")

In [ ]:
# MNIST. ToTensor() gives float32 in [0, 1], shaped (1, 28, 28). No further
# normalisation: the Decoder ends in a sigmoid, so the model's output lives on
# the same scale as its input.
train_set = torchvision.datasets.MNIST(root="./data", train=True,
                                       download=True, transform=transforms.ToTensor())
test_set  = torchvision.datasets.MNIST(root="./data", train=False,
                                       download=True, transform=transforms.ToTensor())

print(f"train {len(train_set)} images | test {len(test_set)} images")
show_grid(torch.stack([train_set[i][0] for i in range(100)]),
          titles=[train_set[i][1] for i in range(100)],
          suptitle="The first 100 MNIST training images")

---

## The model, in one place

$$Pr(x|z,\phi) = \text{Norm}_x\!\left[f[z,\phi],\, \sigma^2 I\right],
\qquad Pr(z) = \text{Norm}_z[0, I],
\qquad q(z|x,\theta) = \text{Norm}_z[\mu, \Sigma]$$

| provided | is | from |
|---|---|---|
| `Encoder()` | $g[x,\theta] \mapsto (\mu,\ \log\sigma^2)$ | `components.py` |
| `Decoder()` | $f[z,\phi] \mapsto \hat{x}$ | `components.py` |
| `reconstruction_loss(x, x_hat)` | $-\log Pr(x \mid z^{*},\phi)$ | 17.23, first term |
| `kl_divergence(mu, logvar)` | $D_{KL}[q(z \mid x,\theta) \,\Vert\, Pr(z)]$ | 17.24 |
| `train_vae(vae, loss_fn, dataset)` | the Lightning training loop | `train.py` |
| `load_z_samples()` | the 100 latent points of Part 4 | `z_samples.pt` |

**The encoder returns $\log\sigma^2$, not $\Sigma$.** A linear layer may output a
negative number and a variance may not, so the network predicts the logarithm and the
code exponentiates it.

**We minimise the negative ELBO.** Chapter 17 maximises the bound; optimisers descend.
Every number printed below is $-\text{ELBO}$, so lower is better, and the KL is
**added** as a positive penalty.

---

# Part 1 — Construct the VAE (30 marks)

Write a `VAE` module that holds one `Encoder` and one `Decoder` and joins them with the
reparameterization.

Three things to write:

**`__init__`** — call `super().__init__()`, then create the two sub-modules. They must
be named **exactly** `self.encoder` and `self.decoder`: `train.encode_dataset` and
`train.decode_z` reach for those attributes, and so does the check.

**`reparameterize(mu, logvar)`** — equation **17.25**:

$$z^{*} = \mu + \Sigma^{1/2}\epsilon^{*},
\qquad \epsilon^{*} \sim \text{Norm}_\epsilon[0, I]$$

$\Sigma$ is diagonal, so $\Sigma^{1/2}$ is the elementwise standard deviation,
$\sigma = \exp\!\left[\tfrac12\log\sigma^2\right]$. Draw $\epsilon^{*}$ with
`torch.randn_like(mu)`, which matches shape, dtype and device in one call.

> `torch.normal(mu, std)` samples the same distribution and **breaks training**: the
> tensor it returns carries no gradient back to `mu`, so the encoder would never learn.
> That is exactly the problem 17.25 exists to solve, and the check tests for it.

**`forward(x)`** — encode, reparameterize, decode. Return the triple
`(x_hat, mu, logvar)`; `train_vae` unpacks all three.

In [ ]:
class VAE(nn.Module):

    def __init__(self, d_z=D_Z):
        super().__init__()
        # YOUR CODE HERE
        raise NotImplementedError()

    def reparameterize(self, mu, logvar):
        # YOUR CODE HERE
        raise NotImplementedError()

    def forward(self, x):
        # YOUR CODE HERE
        raise NotImplementedError()


vae = VAE().to(DEVICE)

In [ ]:
# ----- CHECK, Part 1 ----------------------------------------------------------
check_vae(vae)

---

# Part 2 — Construct the loss (25 marks)

The objective is the single-sample ELBO of **17.23**, negated so it can be descended:

$$-\text{ELBO} \;=\;
\underbrace{-\log\!\left[Pr(x|z^{*},\phi)\right]}_{\texttt{reconstruction\_loss}}
\;+\;
\underbrace{D_{KL}\!\left[q(z|x,\theta) \,\|\, Pr(z)\right]}_{\texttt{kl\_divergence}}$$

Both terms are already written. Call them and add them.

Return the **triple** `(loss, recon, kl)`. `train_vae` logs the two components
separately, which is what gives `plot_metrics` its two panels, so returning the total
alone is not enough.

> Do not reimplement either term. The check compares your values against the provided
> functions and will tell you if they differ.

In [ ]:
def vae_loss(x, x_hat, mu, logvar):
    '''Return (loss, recon, kl), all scalars.'''
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
# ----- CHECK, Part 2 ----------------------------------------------------------
check_loss_fn(vae_loss)

---

# Part 3 — Train it (25 marks)

One call. `train_vae` builds the loader, wraps your model and your loss in a
LightningModule, runs twelve epochs, and returns one row of metrics per epoch:

```python
history = train_vae(vae, vae_loss, train_set)
```

It writes `logs/metrics.csv` and overwrites it on every run, so there is never more
than one copy to look at.

Then call `plot_metrics(history)` to draw the two terms against epoch.

Expect roughly a minute on a GPU. A correct run finishes near `recon` 150 and `kl` 7.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

# history = ...
# plot_metrics(...)

In [ ]:
# ----- CHECK, Part 3 ----------------------------------------------------------
check_history(history)

### **Your answer** — reading the two curves (included in Part 3's marks)

The two terms do not both fall. Say which one **rose**, and explain in two or three
sentences why the model accepts that rise — name what the rising term is buying for the
falling one.

> *Replace this line with your answer.*

---

# Part 4 — Decode the latent sweep (20 marks)

`z_samples.pt` holds 100 latent vectors: a 10×10 sweep of the latent plane, already
ordered for display.

The coordinates are **even in probability, not even in $z$**. The prior is
$\text{Norm}_z[0,I]$, so the ten values on each axis are the midpoints of ten equal
probability slices mapped through the inverse normal CDF —
$\pm1.645, \pm1.036, \pm0.674, \pm0.385, \pm0.126$. A `linspace(-3, 3)` would instead
spend its outer cells in the tails, where the prior puts almost no mass and the model
has seen almost nothing, and those cells decode to smears.

Three steps:

1. `z = load_z_samples()` — a `(100, 2)` tensor.
2. `images = decode_z(vae, z)` — this handles `eval()`, `no_grad()` and the device.
3. `show_grid(images, suptitle=...)` — no per-panel titles here.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

# z = ...
# images = ...
# show_grid(...)

In [ ]:
# ----- CHECK, Part 4 ----------------------------------------------------------
check_decoded_grid(z, images)

### **Your answer** — reading the grid (included in Part 4's marks)

Each cell of the grid is one point of the latent plane. In two or three sentences:

- Name **two digit classes that appear clearly** in your grid and **one that barely
  appears or not at all**.
- The grid covers the middle 90% of the prior in each axis. What would go wrong with
  the pictures if it swept $[-5, 5]$ instead?

> *Replace this line with your answer.*

In [ ]:
# Provided. Saves the trained encoder and decoder for the bonus notebook.
save_trained(vae, "vae.pt")

---

## Marking

| Part | | Marks |
|---|---|---|
| 1 | `VAE` with `self.encoder` and `self.decoder`, a differentiable `reparameterize`, a three-value `forward` | 30 |
| 2 | `vae_loss` returning `(loss, recon, kl)`, built from the provided terms | 25 |
| 3 | the training run and the two-panel plot, plus the answer cell | 25 |
| 4 | the latent sweep loaded, decoded and displayed, plus the answer cell | 20 |
| | **Total** | **100** |

Full marks need every cell **run, with its output showing**. A notebook submitted with
cleared outputs is marked on its code alone.

**On the numbers you should expect.** After twelve epochs a correct run lands near
`recon` 150 and `kl` 7, a total around 157. You are **not** marked on hitting those
figures — training is stochastic and your hardware is not the marker's. You are marked
on having assembled the model correctly, trained it, and read your own output honestly.

**Next:** `assignment3_bonus.ipynb`, worth a further 20 marks.